In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FleetOptimization") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .getOrCreate()

spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d05f1ced-39e2-4a90-b14c-43b94da3a344;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.1.0/delta-spark_2.12-3.1.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.1.0!delta-spark_2.12.jar (624ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.1.0/delta-storage-3.1.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.1.0!delta-storage.jar (235ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (43ms)
:: resolution report :: resolve 2152ms :: artifacts dl 9

In [2]:
from pyspark.sql.functions import col,rand,floor,round,to_timestamp
from pyspark.sql import functions as F
import pandas as pd

In [3]:
import os
print(os.environ["JAVA_HOME"])

/usr/lib/jvm/default-java


26/04/28 16:22:45 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## Test Delta

In [ ]:
spark.range(5).write.format("delta").mode("overwrite").save("/app/test_delta")

In [ ]:
spark.read.format("delta").load("/app/test_delta").show()

In [ ]:
import os
os.listdir("/app/test_delta")

## Raw Parquet to Bronze Layer

In [ ]:
RAW_PATH = '/app/data/raw/'
trips_raw = spark.read.parquet(f"{RAW_PATH}/trips.parquet")
routes_raw = spark.read.parquet(f"{RAW_PATH}/routes.parquet")
drivers_raw = spark.read.parquet(f"{RAW_PATH}/drivers.parquet")
trucks_raw = spark.read.parquet(f"{RAW_PATH}/trucks.parquet")

In [ ]:
BRONZE_PATH = '/app/data/bronze'
trips_raw.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/trips")
routes_raw.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/routes")
drivers_raw.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/drivers")
trucks_raw.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/trucks")

## Bronze to Silver Layer

In [ ]:
from pyspark.sql.functions import col,rand,floor,round
from pyspark.sql import functions as F

In [ ]:
BRONZE_PATH = "/app/data/bronze"
SILVER_PATH = "/app/data/silver"

In [ ]:
trips_df = spark.read.format("delta").load(f"{BRONZE_PATH}/trips")
routes_df = spark.read.format("delta").load(f"{BRONZE_PATH}/routes")

In [ ]:
trips_df.show(1)

In [ ]:
start_date = "2026-01-01"

trips_df = trips_df.withColumn(
    "trip_date",
    F.expr(f"date_add('{start_date}', cast(rand() * 7 as int))")
)

In [ ]:
trips_df = trips_df.withColumn(
    "planned_start_hour",
    (F.rand() * 24).cast("int")
)

trips_df = trips_df.filter(
    (col("distance_km") > 0) &
    (col("load_tons") > 0)
)
routes_trimmed = routes_df.select(
    "route_id",
    "distance_km",
    "avg_speed_kmph",
    "traffic_factor"
)

trips = trips_df.drop("distance_km")  # remove duplicate

joined_df = trips.join(routes_trimmed, on="route_id", how="left")

In [ ]:
joined_df.columns

In [ ]:
joined_df = joined_df.withColumn(
    "estimated_duration_hours",
    round(col("distance_km")/col("avg_speed_kmph")/col("traffic_factor"),2)
)

joined_df = joined_df.withColumn(
    "delivery_deadline_hours",round(
        col("estimated_duration_hours") * (rand() * (1.5 - 1.1) + 1.1),
        2
    )
)

joined_df = joined_df.withColumn(
    "is_deadline_feasible",
    col("estimated_duration_hours") <= col("delivery_deadline_hours")
)

joined_df = joined_df.withColumn(
    "trip_end_hour",
    col("planned_start_hour") + col("estimated_duration_hours")
)

In [ ]:
silver_trips = joined_df.select(
    "trip_id",
    "route_id",
    "source_city",
    "destination_city",
    "distance_km",
    "load_tons",
    "trip_date",                 # include if exists
    "planned_start_hour",
    "trip_end_hour",
    "estimated_duration_hours",
    "delivery_deadline_hours",
    "is_deadline_feasible"
)

In [ ]:
silver_trips.write.format("delta").mode("overwrite").save("/app/data/silver/trips")

## Assignment Engine : Version 1 ( Selecting the combination based on Cost , Drivers based on drivinghours_per_day for trip vs 

In [ ]:
trips_df = spark.read.format("delta").load("/app/data/silver/trips")
trucks_df = spark.read.format("delta").load("/app/data/bronze/trucks")
drivers_df = spark.read.format("delta").load("/app/data/bronze/drivers")

In [ ]:
valid_trucks.columns

In [ ]:
valid_trucks = trucks_df.filter(col("status")=="available")

In [ ]:
trip_truck = trips_df.join(valid_trucks,trips_df.load_tons<=valid_trucks.capacity_tons,how="inner")

In [ ]:
from pyspark.sql.functions import when

FUEL_PRICE = 100
DRIVER_RATE = 200
LATE_PENALTY = 500

trip_truck = trip_truck.withColumn(
    "fuel_cost",
    (col("distance_km") / col("fuel_efficiency_kmpl")) * FUEL_PRICE
)

trip_truck = trip_truck.withColumn(
    "driver_cost",
    col("estimated_duration_hours") * DRIVER_RATE
)

trip_truck = trip_truck.withColumn(
    "delay",
    col("estimated_duration_hours") - col("delivery_deadline_hours")
)

trip_truck = trip_truck.withColumn(
    "penalty",
    when(col("delay") > 0, col("delay") * LATE_PENALTY).otherwise(0)
)

trip_truck = trip_truck.withColumn(
    "total_cost",
    col("fuel_cost") + col("driver_cost") + col("penalty")
)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,ceil

In [ ]:
window_truck = Window.partitionBy("trip_id").orderBy("total_cost")
best_truck = trip_truck.withColumn("truck_rank",row_number().over(window_truck)).filter(col("truck_rank")==1)

In [ ]:
trip_truck_driver = best_truck.crossJoin(drivers_df)

In [ ]:
trip_truck_driver = trip_truck_driver.withColumn(
    "required_days",
    ceil(col("estimated_duration_hours") / col("max_hours_per_day"))
)

trip_truck_driver = trip_truck_driver.withColumn(
    "hours_per_day",
    col("estimated_duration_hours") / col("required_days")
)

trip_truck_driver = trip_truck_driver.filter(
    col("hours_per_day") <= col("max_hours_per_day")
)

In [ ]:
window_driver = Window.partitionBy("trip_id").orderBy("total_cost")

final_assignments = trip_truck_driver.withColumn(
    "driver_rank",
    row_number().over(window_driver)
).filter(col("driver_rank") == 1)

In [ ]:
assignments = final_assignments.select(
    "trip_id",
    "truck_id",
    "driver_id",
    "route_id",
    "planned_start_hour",
    "trip_end_hour",
    "total_cost"
)

In [ ]:
assignments.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/app/data/silver/assignments")

## Trip Level Optimization to Assignment or Scheduling Optimization

In [ ]:
### To Assign only one trip for one driver per day

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

driver_window = Window.partitionBy("driver_id").orderBy("total_cost")

unique_driver_assignments = final_assignments.withColumn(
    "driver_trip_rank",
    row_number().over(driver_window)
).filter(col("driver_trip_rank") == 1)

In [ ]:
assigned_trip_ids = unique_driver_assignments.select("trip_id")

In [ ]:
unassigned_trips = trips_df.join(
    assigned_trip_ids,
    on="trip_id",
    how="left_anti"
)

In [ ]:
assigned_trip_ids.show()

In [ ]:
final_assignments.count()

## Assignment Engine : Version 2 ( Selecting the combination based on Cost , Driver Schedule)

In [5]:
trips_df = spark.read.format("delta").load("/app/data/bronze/trips")
routes_df = spark.read.format("delta").load("/app/data/bronze/routes")
trucks_df = spark.read.format("delta").load("/app/data/bronze/trucks")
drivers_df = spark.read.format("delta").load("/app/data/bronze/drivers")

In [8]:
start_date = "2026-01-01"
trips_df = trips_df.withColumn(
    "trip_date",
    F.expr(f"date_add('{start_date}', cast(rand() * 7 as int))")
)

trips_df = trips_df.withColumn(
    "planned_start_hour",
    (F.rand() * 24).cast("int")
)

routes_df = routes_df.select('route_id','avg_speed_kmph','traffic_factor')

joined_df = trips_df.join(
    routes_df,
    on="route_id",
    how="left"
)

In [9]:
joined_df.show(1)

26/04/26 14:52:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------+-------+-----------+----------------+-----------+---------+-----------------------+----------+------------------+--------------+--------------+
|route_id|trip_id|source_city|destination_city|distance_km|load_tons|delivery_deadline_hours| trip_date|planned_start_hour|avg_speed_kmph|traffic_factor|
+--------+-------+-----------+----------------+-----------+---------+-----------------------+----------+------------------+--------------+--------------+
|      20|      1|     Mumbai|           Delhi|       1506|        2|                     31|2026-01-03|                 6|            59|          0.97|
+--------+-------+-----------+----------------+-----------+---------+-----------------------+----------+------------------+--------------+--------------+
only showing top 1 row



In [10]:
from pyspark.sql.functions import col, expr,unix_timestamp

joined_df = joined_df.withColumn(
    "estimated_duration_hours",
    round(col("distance_km")/col("avg_speed_kmph")/col("traffic_factor"),2)
)

joined_df = joined_df.withColumn(
    "trip_timestamp",
    to_timestamp(col("trip_date"))
)

joined_df = joined_df.withColumn(
    "planned_start_time",
    col("trip_timestamp") + expr("INTERVAL 1 HOUR") * col("planned_start_hour")
)

joined_df = joined_df.withColumn(
    "planned_start_time",
    (round(unix_timestamp(col("planned_start_time")) / 1800) * 1800).cast("timestamp")
)


joined_df = joined_df.withColumn(
    "trip_end_time",
    col("planned_start_time") + expr("INTERVAL 1 HOUR") * col("estimated_duration_hours")
)

joined_df = joined_df.withColumn(
    "trip_end_time",
    (round(unix_timestamp(col("trip_end_time")) / 1800) * 1800).cast("timestamp")
)


joined_df = joined_df.withColumn(
    "next_available_time",
    col("trip_end_time") + expr("INTERVAL 1 HOUR") * 24
)



In [11]:
trips_df = joined_df

In [12]:
from pyspark.sql.functions import col

trip_truck = trips_df.join(
    trucks_df,
    trips_df.load_tons <= trucks_df.capacity_tons,
    "inner"
)

In [13]:
trip_truck = trip_truck.withColumn(
    "fuel_cost",
    (col("distance_km") / col("fuel_efficiency_kmpl")) * 100
)

trip_truck = trip_truck.withColumn(
    "driver_cost",
    col("estimated_duration_hours") * 200
)

trip_truck = trip_truck.withColumn(
    "total_cost",
    col("fuel_cost") + col("driver_cost")
)

In [18]:
candidate_df.count()

200000000

### Limit Trucks per Trip

In [15]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

truck_window = Window.partitionBy("trip_id").orderBy("fuel_cost")

trip_truck = trip_truck.withColumn(
    "truck_rank",
    row_number().over(truck_window)
).filter(col("truck_rank") <= 2)

In [17]:
candidate_df = trip_truck.crossJoin(drivers_df)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("trip_id").orderBy("total_cost")

candidate_df = candidate_df.withColumn(
    "rank",
    row_number().over(window)
).filter(col("rank") <= 3)

In [ ]:
candidate_df = candidate_df.orderBy("planned_start_time", "total_cost")

In [ ]:
candidate_df.count()

## Assignment Engine : Version 3 ( Selecting the combination based on Cost , Driver Schedule , Handling Performance Bottleneck issues)

In [5]:
trips_df = spark.read.format("delta").load("/app/data/bronze/trips")
routes_df = spark.read.format("delta").load("/app/data/bronze/routes")
trucks_df = spark.read.format("delta").load("/app/data/bronze/trucks")
drivers_df = spark.read.format("delta").load("/app/data/bronze/drivers")

In [6]:
from pyspark.sql.functions import col, expr,unix_timestamp,count

In [7]:
start_date = "2026-01-01"
trips_df = trips_df.withColumn(
    "trip_date",
    F.expr(f"date_add('{start_date}', cast(rand() * 7 as int))")
)

trips_df = trips_df.withColumn(
    "planned_start_hour",
    (F.rand() * 24).cast("int")
)

routes_df = routes_df.select('route_id','avg_speed_kmph','traffic_factor')

joined_df = trips_df.join(
    routes_df,
    on="route_id",
    how="left"
)

In [8]:


joined_df = joined_df.withColumn(
    "estimated_duration_hours",
    round(col("distance_km")/col("avg_speed_kmph")/col("traffic_factor"),2)
)

joined_df = joined_df.withColumn(
    "trip_timestamp",
    to_timestamp(col("trip_date"))
)

joined_df = joined_df.withColumn(
    "planned_start_time",
    col("trip_timestamp") + expr("INTERVAL 1 HOUR") * col("planned_start_hour")
)

joined_df = joined_df.withColumn(
    "planned_start_time",
    (round(unix_timestamp(col("planned_start_time")) / 1800) * 1800).cast("timestamp")
)


joined_df = joined_df.withColumn(
    "trip_end_time",
    col("planned_start_time") + expr("INTERVAL 1 HOUR") * col("estimated_duration_hours")
)

joined_df = joined_df.withColumn(
    "trip_end_time",
    (round(unix_timestamp(col("trip_end_time")) / 1800) * 1800).cast("timestamp")
)


joined_df = joined_df.withColumn(
    "next_available_time",
    col("trip_end_time") + expr("INTERVAL 1 HOUR") * 24
)



In [9]:
trips_df = joined_df
trips_df.limit(5).toPandas()

26/04/28 16:28:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

,route_id,trip_id,source_city,destination_city,distance_km,load_tons,delivery_deadline_hours,trip_date,planned_start_hour,avg_speed_kmph,traffic_factor,estimated_duration_hours,trip_timestamp,planned_start_time,trip_end_time,next_available_time
0,20,1,Mumbai,Delhi,1506,2,31,2026-01-06,1,59,0.97,26.31,2026-01-06,2026-01-06 01:00:00,2026-01-07 03:30:00,2026-01-08 03:30:00
1,2,2,Bangalore,Hyderabad,528,19,21,2026-01-02,12,75,1.18,5.97,2026-01-02,2026-01-02 12:00:00,2026-01-02 18:00:00,2026-01-03 18:00:00
2,26,3,Delhi,Bangalore,2116,24,16,2026-01-02,11,53,1.45,27.53,2026-01-02,2026-01-02 11:00:00,2026-01-03 14:30:00,2026-01-04 14:30:00
3,30,4,Delhi,Pune,1323,16,16,2026-01-07,21,41,1.21,26.67,2026-01-07,2026-01-07 21:00:00,2026-01-08 23:30:00,2026-01-09 23:30:00
4,30,5,Delhi,Pune,1323,5,39,2026-01-07,21,41,1.21,26.67,2026-01-07,2026-01-07 21:00:00,2026-01-08 23:30:00,2026-01-09 23:30:00


In [10]:
trucks_pd = trucks_df.toPandas()
drivers_pd = drivers_df.toPandas()

driver_ids = drivers_pd["driver_id"].tolist()

In [30]:
TOP_K_DRIVERS = 5
TOP_M_TRUCKS = 3

FUEL_PRICE = 100
DRIVER_RATE = 200
REST_HOURS = 8

In [31]:
driver_next_available = {}   # persists across days
all_assignments = []
all_unassigned = []

In [13]:
def get_available_drivers(start_time):
    return [
        d for d in driver_ids
        if d not in driver_next_available or start_time >= driver_next_available[d]
    ]

In [14]:
def get_top_trucks(load):
    valid = trucks_pd[trucks_pd["capacity_tons"] >= load]
    
    if valid.empty:
        return valid
    
    return valid.sort_values(
        "fuel_efficiency_kmpl", ascending=False
    ).head(TOP_M_TRUCKS)

In [15]:
def compute_cost(trip, truck):
    fuel_cost = (trip["distance_km"] / truck["fuel_efficiency_kmpl"]) * FUEL_PRICE
    driver_cost = trip["estimated_duration_hours"] * DRIVER_RATE
    return fuel_cost + driver_cost

In [32]:
dates = [
    row["trip_date"]
    for row in trips_df.select("trip_date").distinct().collect()
]

dates = sorted(dates)

In [33]:
for date in dates:

    print(f"\nProcessing date: {date}")

    # Load only that day's trips
    trips_pd = trips_df.filter(col("trip_date") == date) \
                       .orderBy("planned_start_time") \
                       .toPandas()

    print(f"Trips count: {len(trips_pd)}")

    # --- Trip loop ---
    for _, trip in trips_pd.iterrows():

        trip_id = trip["trip_id"]
        start = trip["planned_start_time"]
        end = trip["trip_end_time"]
        load = trip["load_tons"]

        # STEP 1 — available drivers
        available = get_available_drivers(start)

        if not available:
            all_unassigned.append({
                "trip_id": trip_id,
                "reason": "No available drivers"
            })
            continue

        selected_drivers = available[:TOP_K_DRIVERS]

        # STEP 2 — trucks
        top_trucks = get_top_trucks(load)

        if top_trucks.empty:
            all_unassigned.append({
                "trip_id": trip_id,
                "reason": "No valid trucks"
            })
            continue

        # STEP 3 — evaluate
        best_cost = float("inf")
        best_driver = None
        best_truck = None

        for driver in selected_drivers:
            for _, truck in top_trucks.iterrows():

                cost = compute_cost(trip, truck)

                if cost < best_cost:
                    best_cost = cost
                    best_driver = driver
                    best_truck = truck["truck_id"]

        # STEP 4 — assign
        all_assignments.append({
            "trip_id": trip_id,
            "trip_date": date,
            "driver_id": best_driver,
            "truck_id": best_truck,
            "planned_start_time": start,
            "trip_end_time": end,
            "total_cost": best_cost
        })

        # STEP 5 — update driver state (CRITICAL)
        driver_next_available[best_driver] = end + pd.Timedelta(hours=REST_HOURS)


Processing date: 2026-01-01
Trips count: 14297

Processing date: 2026-01-02
Trips count: 14260

Processing date: 2026-01-03
Trips count: 14284

Processing date: 2026-01-04
Trips count: 14180

Processing date: 2026-01-05
Trips count: 14369

Processing date: 2026-01-06
Trips count: 14289

Processing date: 2026-01-07
Trips count: 14321


In [34]:
assignments_df = pd.DataFrame(all_assignments)
assignments_df.shape

(7850, 7)

In [36]:
drviers_trips = assignments_df.groupby("driver_id")["trip_id"].count()
drviers_trips

driver_id
1        7
2        9
3        7
4        7
5        9
        ..
996      7
997     10
998      9
999      8
1000     6
Name: trip_id, Length: 1000, dtype: int64

In [37]:
unassigned_df = pd.DataFrame(all_unassigned)
unassigned_df.shape

(92150, 2)

In [38]:
unassigned_df.head(1)

,trip_id,reason
0,64324,No available drivers


In [45]:
spark.createDataFrame(assignments_df) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save("/app/data/silver/final_assignments")

26/04/26 17:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/26 17:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/26 17:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/26 17:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/26 17:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/26 17:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/26 17:09:47 WARN MemoryManager: Total allocation exceeds 95.

In [46]:
spark.createDataFrame(unassigned_df) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save("/app/data/silver/unassigned_trips")

26/04/26 17:09:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/26 17:09:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/26 17:09:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/26 17:09:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/26 17:09:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/26 17:09:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/26 17:09:57 WARN MemoryManager: Total allocation exceeds 95.

### Reducing the trips_df and rerunning the logic

In [53]:
filtered_trips = trips_df.filter(
    col("estimated_duration_hours") <= col("delivery_deadline_hours")
)
filtered_trips = filtered_trips.limit(8000)

In [54]:
trucks_pd = trucks_df.toPandas()
drivers_pd = drivers_df.toPandas()

driver_ids = drivers_pd["driver_id"].tolist()

TOP_K_DRIVERS = 5
TOP_M_TRUCKS = 3

FUEL_PRICE = 100
DRIVER_RATE = 200
REST_HOURS = 8

driver_next_available = {}   # persists across days
all_assignments = []
all_unassigned = []

dates = [
    row["trip_date"]
    for row in trips_df.select("trip_date").distinct().collect()
]

dates = sorted(dates)

In [55]:
for date in dates:

    print(f"\nProcessing date: {date}")

    # Load only that day's trips
    trips_pd = filtered_trips.filter(col("trip_date") == date) \
                       .orderBy("planned_start_time") \
                       .toPandas()

    print(f"Trips count: {len(trips_pd)}")

    # --- Trip loop ---
    for _, trip in trips_pd.iterrows():

        trip_id = trip["trip_id"]
        start = trip["planned_start_time"]
        end = trip["trip_end_time"]
        load = trip["load_tons"]

        # STEP 1 — available drivers
        available = get_available_drivers(start)

        if not available:
            all_unassigned.append({
                "trip_id": trip_id,
                "reason": "No available drivers"
            })
            continue

        selected_drivers = available[:TOP_K_DRIVERS]

        # STEP 2 — trucks
        top_trucks = get_top_trucks(load)

        if top_trucks.empty:
            all_unassigned.append({
                "trip_id": trip_id,
                "reason": "No valid trucks"
            })
            continue

        # STEP 3 — evaluate
        best_cost = float("inf")
        best_driver = None
        best_truck = None

        for driver in selected_drivers:
            for _, truck in top_trucks.iterrows():

                cost = compute_cost(trip, truck)

                if cost < best_cost:
                    best_cost = cost
                    best_driver = driver
                    best_truck = truck["truck_id"]

        # STEP 4 — assign
        all_assignments.append({
            "trip_id": trip_id,
            "trip_date": date,
            "driver_id": best_driver,
            "truck_id": best_truck,
            "planned_start_time": start,
            "trip_end_time": end,
            "total_cost": best_cost
        })

        # STEP 5 — update driver state (CRITICAL)
        driver_next_available[best_driver] = end + pd.Timedelta(hours=REST_HOURS)


Processing date: 2026-01-01
Trips count: 1119

Processing date: 2026-01-02
Trips count: 1159

Processing date: 2026-01-03
Trips count: 1202

Processing date: 2026-01-04
Trips count: 1112

Processing date: 2026-01-05
Trips count: 1143

Processing date: 2026-01-06
Trips count: 1140

Processing date: 2026-01-07
Trips count: 1125


In [56]:
assignments_df = pd.DataFrame(all_assignments)
assignments_df.shape

(7947, 7)

In [57]:
unassigned_df = pd.DataFrame(all_unassigned)
unassigned_df.shape

(53, 2)

In [58]:
unassigned_df

,trip_id,reason
0,9244,No available drivers
1,9382,No available drivers
2,9463,No available drivers
3,9610,No available drivers
4,9631,No available drivers
5,10028,No available drivers
6,10103,No available drivers
7,10338,No available drivers
8,10448,No available drivers
9,10557,No available drivers
